In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
!pip install catboost

from catboost import CatBoostClassifier


In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

df = pd.read_csv(f"{path}/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()


In [ ]:
# Task 4: Write your code here:
df.describe()


In [ ]:
# Task 1: Write your code here:
# Task 2: Write your code here:

#to check what type of misssing values we have
# 2. Do we have missing values?
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(df)

print("Missing values before handeling:", df.isnull().sum().sum())  #output = 311

#handeling them:
#simply any numarical null fill with median
#any categorical fill with mode the most repeated
num_cols_all = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols_all = df.select_dtypes(include=["object", "category"]).columns

for col in num_cols_all:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols_all:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after handeling:", df.isnull().sum().sum())  #output = 311


In [ ]:
# Task 2: Write your code here:

# 4. Do we have duplicate samples?
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(df)

df = df.drop_duplicates()
after_dupes = df.duplicated().sum()
print(f"after drop: {after_dupes}")


In [ ]:
# Task 3: Write your code here:
cat_cols = df.select_dtypes(include=["object", "category"]).columns
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

In [ ]:
# Task 4: Write your code here:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [ ]:
# Task 5: Write your code here:
target_ratio = df["Target"].value_counts(normalize=True)
print("Target distribution:\n", target_ratio)
print("it is imbalanced so ganna use F1-score")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df["Target"]


In [ ]:
# Task 2,3,4,5: Write your code here:
#ganna use StratifiedKFold because it is imbalanced
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=40,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=False
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    f1 = f1_score(y_val, preds)
    f1_scores.append(f1)
    print(f"Fold {fold} F1-score: {f1:.4f}")

print(f"\nAverage F1-score across folds: {np.mean(f1_scores):.4f}")


In [ ]:
# Task 1: Write your code here:
model.fit(X, y)

importances = model.get_feature_importance()
feature_names = X.columns

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi.head(20), x="importance", y="feature")
plt.title("Top 20 Feature Importances (CatBoost)")
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:
golden_feature = fi.iloc[0]["feature"]
print("Golden Feature:", golden_feature)

In [ ]:
# Task Bonus: Write your code here:
X_golden = X[[golden_feature]]
f1_scores_golden = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_golden, y), start=1):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_golden = CatBoostClassifier(
        iterations=40,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=False
    )

    model_golden.fit(X_train, y_train)
    preds = model_golden.predict(X_val)

    f1 = f1_score(y_val, preds)
    f1_scores_golden.append(f1)
    print(f"Fold {fold} Golden Feature F1-score: {f1:.4f}")

print(f"\nAverage Golden Feature F1-score: {np.mean(f1_scores_golden):.4f}")